<a href="https://www.kaggle.com/code/faihaj/random-100-hosts-for-training-optc-dataset?scriptVersionId=307380330" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install huggingface_hub

In [2]:
import os
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_API_TOKEN")
os.environ["HF_TOKEN"] = user_secrets.get_secret("kaggle_hf_key")
api = HfApi()
print(f"Logged in as: {api.whoami()['name']}")

Logged in as: equalNull


## find the files common every day
let us find the hosts present in every day.

In [3]:
import re
from collections import Counter
from huggingface_hub import HfApi

api = HfApi()
total_counts = Counter()

# 1. Iterate through all 10 repositories
for day in range(16, 26):
    repo_id = f"equalNull/labelled-optc-sep{day}"
    print(f"Scanning {repo_id}...")
    
    try:
        # Get the list of all file names in the repo
        files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
        
        # 2. Extract 4-digit numbers from filenames
        for filename in files:
            # Finds all 4-digit sequences in the filename
            matches = re.findall(r'\d{4}', filename)
            for match in matches:
                # Only count if it's within your 0000-1000 range
                if 0 <= int(match) <= 1000:
                    total_counts[match] += 1
                    
    except Exception as e:
        print(f"Skipping {repo_id} due to error: {e}")


Scanning equalNull/labelled-optc-sep16...
Scanning equalNull/labelled-optc-sep17...
Scanning equalNull/labelled-optc-sep18...
Scanning equalNull/labelled-optc-sep19...
Scanning equalNull/labelled-optc-sep20...
Scanning equalNull/labelled-optc-sep21...
Scanning equalNull/labelled-optc-sep22...
Scanning equalNull/labelled-optc-sep23...
Scanning equalNull/labelled-optc-sep24...
Scanning equalNull/labelled-optc-sep25...


In [4]:
# Filter for IDs that appeared in all 10 repositories
all_10_days = [p for p, count in total_counts.items() if count == 10]
print(all_10_days)
print(len(all_10_days))

['0101', '0102', '0103', '0104', '0105', '0106', '0107', '0108', '0109', '0110', '0111', '0112', '0113', '0114', '0115', '0116', '0117', '0118', '0119', '0120', '0121', '0122', '0123', '0124', '0125', '0151', '0152', '0153', '0154', '0155', '0156', '0157', '0158', '0159', '0160', '0161', '0162', '0163', '0164', '0165', '0166', '0167', '0168', '0169', '0170', '0171', '0172', '0173', '0174', '0175', '0201', '0202', '0203', '0204', '0205', '0206', '0207', '0208', '0209', '0210', '0211', '0212', '0214', '0215', '0216', '0217', '0218', '0219', '0220', '0221', '0223', '0224', '0225', '0251', '0252', '0253', '0254', '0255', '0256', '0257', '0258', '0259', '0260', '0261', '0262', '0263', '0264', '0265', '0266', '0267', '0268', '0269', '0270', '0271', '0272', '0273', '0274', '0275', '0301', '0302', '0303', '0304', '0305', '0306', '0307', '0308', '0309', '0310', '0311', '0312', '0313', '0314', '0315', '0316', '0317', '0318', '0319', '0320', '0321', '0322', '0323', '0324', '0325', '0351', '0352',

## hosts attacked during attack phase
### on sep 23
* SysClient0104
* SysClient0321
* SysClient0355
* SysClient0660
* SysClient0201
* SysClient0609
* SysClient0874
* SysClient0955
* SysClient0205
* SysClient0255
* SysClient0402
* SysClient0419
* SysClient0462
* SysClient0170
* SysClient0503
* SysClient0559
* SysClient0771
### on sep 24
* SysClient0501
* SysClient0069
* SysClient0811
* SysClient0974
* SysClient0203
* SysClient0358
* SysClient0618
* SysClient0005
* SysClient0851
* SysClient0010
### on sep 25
* SysClient0051
* SysClient0351
* SysClient0203
* SysClient0358
* SysClient0851
* SysClient0010
* SysClient0069
* SysClient0501
* SysClient0618


```python
mal_hosts = {"0104", "0321", "0355", "0660", "0201", "0609", "0874", "0955", "0205", "0255", "0402", "0419", "0462", "0170", "0503", "0559", "0771", "0501", "0069", "0811", "0974", "0203", "0358", "0618", "0005", "0851", "0010", "0051", "0351", "0203", "0358", "0851", "0010", "0069", "0501", "0618"}
```
The proof of this can be found [here](https://www.kaggle.com/code/faihaj/labelling-optc-dataset?scriptVersionId=306235992).

## generate names of random host patterns

In [5]:
import random

# 1. Start with known malicious hosts
hosts = {"0104", "0321", "0355", "0660", "0201", "0609", "0874", "0955", 
         "0205", "0255", "0402", "0419", "0462", "0170", "0503", "0559", 
         "0771", "0501", "0069", "0811", "0974", "0203", "0358", "0618", 
         "0005", "0851", "0010", "0051", "0351", "0203", "0358", "0851", 
         "0010", "0069", "0501", "0618"}

# 2. Identify how many more you need to reach 100
needed = 100 - len(hosts)

print(f"Will need the extra {needed} hosts")
# 3. Filter all_10_days to exclude what's already in hosts
candidates = [p for p in all_10_days if p not in hosts]

# 4. Randomly pick from the persistent set instead of a random range
if len(candidates) >= needed:
    new_hosts = random.sample(candidates, needed)
else:
    # Fallback if there aren't enough persistent IDs
    print(f"Warning: Only {len(candidates)} candidates found in all_10_days. Using all of them.")
    new_hosts = candidates

# 5. Combine and create patterns
hosts.update(new_hosts)
host_patterns = {f"*{h}*" for h in hosts}

print(f"Final host count: {len(hosts)}")


Will need the extra 71 hosts
Final host count: 100


In [6]:
print(host_patterns)

{'*0124*', '*0660*', '*0168*', '*0075*', '*0462*', '*0466*', '*0318*', '*0005*', '*0771*', '*0272*', '*0121*', '*0452*', '*0456*', '*0404*', '*0303*', '*0259*', '*0322*', '*0424*', '*0203*', '*0475*', '*0364*', '*0060*', '*0010*', '*0255*', '*0107*', '*0367*', '*0054*', '*0467*', '*0225*', '*0258*', '*0451*', '*0402*', '*0217*', '*0501*', '*0169*', '*0618*', '*0172*', '*0261*', '*0275*', '*0974*', '*0470*', '*0069*', '*0162*', '*0056*', '*0874*', '*0355*', '*0955*', '*0154*', '*0212*', '*0269*', '*0315*', '*0419*', '*0161*', '*0065*', '*0454*', '*0559*', '*0201*', '*0170*', '*0152*', '*0370*', '*0105*', '*0411*', '*0410*', '*0851*', '*0051*', '*0125*', '*0111*', '*0063*', '*0251*', '*0321*', '*0356*', '*0307*', '*0469*', '*0109*', '*0503*', '*0104*', '*0416*', '*0406*', '*0362*', '*0103*', '*0323*', '*0205*', '*0316*', '*0358*', '*0421*', '*0252*', '*0453*', '*0204*', '*0151*', '*0609*', '*0207*', '*0102*', '*0167*', '*0304*', '*0811*', '*0472*', '*0351*', '*0314*', '*0053*', '*0214*'}

## Download the parquets
We will download all the hosts parquet for each day during `16th of september` to `25th of september`.
## upload into hf dataset

In [7]:
from huggingface_hub import snapshot_download

new_repo_id = f"equalNull/hosts100-labelled-optc"
api.create_repo(repo_id=new_repo_id, repo_type="dataset", exist_ok=True)
working_path = "/kaggle/temp/"
os.makedirs(working_path, exist_ok=True)
for day in range(16,26):
    repo_id = f"equalNull/labelled-optc-sep{day}"
    downloaded_path = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=working_path,
        allow_patterns=list(host_patterns),
        max_workers=4
    )
    
    !mkdir -p {working_path}data/sep{day}
    !mv {working_path}data/*.parquet {working_path}data/sep{day}/
    api.upload_large_folder(
        folder_path=f"{working_path}data/",
        repo_id=new_repo_id,
        repo_type="dataset"
    )
    !rm -rf {working_path}data/

Fetching 86 files:   0%|          | 0/86 [00:00<?, ?it/s]

Recovering from metadata files:   0%|          | 0/86 [00:00<?, ?it/s]




---------- 2026-03-29 08:17:34 (0:00:00) ----------
Files:   hashed 0/86 (0.0/1.8G) | pre-uploaded: 0/0 (0.0/1.8G) (+86 unsure) | committed: 0/86 (0.0/1.8G) | ignored: 0
Workers: hashing: 2 | get upload mode: 0 | pre-uploading: 0 | committing: 0 | waiting: 0
---------------------------------------------------


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


---------- 2026-03-29 09:08:28 (0:01:00) ----------
Files:   hashed 100/100 (6.9G/6.9G) | pre-uploaded: 100/100 (6.9G/6.9G) | committed: 50/100 (3.5G/6.9G) | ignored: 0
Workers: hashing: 0 | get upload mode: 0 | pre-uploading: 0 | committing: 1 | waiting: 1
---------------------------------------------------
                             